# Heart Disease Prediction & Classification
This capstone project focuses on building, tuning, and evaluating machine learning models to predict the presence of heart disease in patients. The dataset consists of various physiological measurements (like age, resting blood pressure, serum cholesterol, max heart rate) and categorical variables (like sex, chest pain type, exercise induced angina).

### Objective & Metric Justification
In medical diagnostics, **minimizing False Negatives** is the highest priority. A false negative means a patient with heart disease is incorrectly classified as healthy, leading to delayed treatment and potentially fatal outcomes. A false positive (labeling a healthy patient as sick) will lead to further testing but is far less hazardous. 
Therefore, our primary optimization metric for hyperparameter tuning is **Recall (Sensitivity) for Class 1 (Heart Disease Presence)**.


In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import boxcox

import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

premium_colors = ['#4F46E5', '#F43F5E', '#0EA5E9', '#10B981', '#F59E0B', '#8B5CF6', '#EC4899', '#14B8A6', '#F97316']

import plotly.io as pio
pio.renderers.default = 'notebook_connected'
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)


## 1. Data Loading & Inspection
We load the dataset `heart.csv` and inspect its initial structure.


In [38]:
df = pd.read_csv("heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


## 2. Variable Separation & Type Casting
We identify the categorical and continuous variables. To ensure correct summary statistics and appropriate preprocessing, we convert the categorical columns to Python `object` type.


In [40]:
columns= ['sex','cp','fbs','restecg','exang','slope','ca','thal','target']
continuous = [feature for feature in df.columns if feature not in columns]
df[columns] = df[columns].astype('object')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    object 
 2   cp        303 non-null    object 
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    object 
 6   restecg   303 non-null    object 
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    object 
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    object 
 11  ca        303 non-null    object 
 12  thal      303 non-null    object 
 13  target    303 non-null    object 
dtypes: float64(1), int64(4), object(9)
memory usage: 33.3+ KB


In [41]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif = pd.DataFrame()
vif["Feature"] = continuous
vif["VIF"] = [variance_inflation_factor(df[continuous].values, i) for i in range(len(continuous))]
vif

,Feature,VIF
0,age,36.410374
1,trestbps,55.751559
2,chol,24.228662
3,thalach,29.378507
4,oldpeak,2.093841


* **VIF Interpretation:**
  * A VIF value **> 10** generally indicates high multicollinearity, while **> 5** is considered moderate.
  * Here, `trestbps` (60.78), `age` (37.64), `thalach` (30.71), and `chol` (27.85) all show elevated VIF values. This is partly expected because these physiological measurements are naturally correlated (e.g., age influences blood pressure and max heart rate).
  * However, `oldpeak` has a healthy VIF of **2.15**, indicating it provides largely independent information.
  * **Note:** Since our primary models include tree-based methods (Decision Trees, Random Forest, Gradient Boosting), which are inherently **immune to multicollinearity**, this does not negatively impact their predictions. For linear models or SVM, the `Pipeline` with `StandardScaler` helps mitigate scaling effects, though multicollinearity can still inflate coefficient variance.


In [42]:
df.describe()

,age,trestbps,chol,thalach,oldpeak
count,303.000000,303.000000,303.000000,303.000000,303.000000
mean,54.366337,131.623762,246.264026,149.646865,1.039604
std,9.082101,17.538143,51.830751,22.905161,1.161075
min,29.000000,94.000000,126.000000,71.000000,0.000000
25%,47.500000,120.000000,211.000000,133.500000,0.000000
50%,55.000000,130.000000,240.000000,153.000000,0.800000
75%,61.000000,140.000000,274.500000,166.000000,1.600000
max,77.000000,200.000000,564.000000,202.000000,6.200000


In [43]:
df.describe(include="object")

,sex,cp,fbs,restecg,exang,slope,ca,thal,target
count,303,303,303,303,303,303,303,303,303
unique,2,4,2,3,2,3,5,4,2
top,1,0,0,1,0,2,0,2,1
freq,207,143,258,152,204,142,175,166,165


## 3. Exploratory Data Analysis (EDA)

### 3.1. Distributions of Continuous Features
We plot the distribution of each continuous variable using histograms overlayed with a Kernel Density Estimate (KDE) to visualize shape, skewness, and spread.


In [44]:
"""
(Main Code)
plt.figure(figsize=(15, 10))
for i, col in enumerate(continuous, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], kde=True, color='royalblue')
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()
"""

# ── Continuous Feature Distributions: Histogram + KDE (PLOTLY PRO VERSION) ──
df_continuous = df[continuous]
col_labels = {
    'age': 'Age (years)',
    'trestbps': 'Resting Blood Pressure (mmHg)',
    'chol': 'Serum Cholesterol (mg/dl)',
    'thalach': 'Max Heart Rate Achieved (bpm)',
    'oldpeak': 'ST Depression (Oldpeak)'
}

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[col_labels.get(c, c) for c in df_continuous.columns],
    vertical_spacing=0.14,
    horizontal_spacing=0.08
)

for i, col in enumerate(df_continuous.columns):
    row = (i // 3) + 1
    col_idx = (i % 3) + 1
    color = premium_colors[i % len(premium_colors)]

    dist_fig = ff.create_distplot(
        [df_continuous[col].dropna().tolist()],
        [col_labels.get(col, col)],
        show_rug=False,
        colors=[color],
        bin_size=(df_continuous[col].max() - df_continuous[col].min()) / 25
    )
    for trace in dist_fig['data']:
        if trace['type'] == 'histogram':
            trace['opacity'] = 0.45
            trace['marker']['line'] = dict(color='white', width=0.5)
        elif trace['type'] == 'scatter':  # KDE line
            trace['line'] = dict(width=2.5, color=color)
        fig.add_trace(trace, row=row, col=col_idx)

# Hide the unused 6th subplot
fig.update_layout(
    height=640,
    width=1150,
    title_text='<b>Continuous Feature Distributions</b><br><sup>Histogram with Kernel Density Estimate</sup>',
    showlegend=False,
    margin=dict(l=50, r=30, t=100, b=50)
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9')

for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=13, color='#334155', family='Inter, Arial')

fig.show()


* **Key Observations:**
  * **Age:** Approximately normally distributed, peaking between 50 and 60 years.
  * **Trestbps (Resting Blood Pressure):** Slightly right-skewed, with most patients having values between 120 and 140 mmHg.
  * **Chol (Serum Cholesterol):** Right-skewed, showing an outlier spike beyond 400 mg/dl.
  * **Thalach (Max Heart Rate Achieved):** Left-skewed, centered around 150 bpm.
  * **Oldpeak (ST Depression):** Strongly right-skewed, with a large number of patients having 0 ST depression.


### 3.2. Distributions of Categorical Features
We check the counts and relative percentages of all categorical categories, including our target variable.


In [45]:
"""
(Main Code)
plt.figure(figsize=(15, 15))
for i, col in enumerate(columns, 1):
    plt.subplot(3, 3, i)
    sns.countplot(y=df[col], palette='viridis')
    plt.title(f'Counts for {col}')
plt.tight_layout()
plt.show()
"""

# ── Categorical Feature Distributions: Horizontal Bar Charts (PLOTLY) ───────
df_categorical = df[columns]

cat_labels = {
    'sex':     'Sex',
    'cp':      'Chest Pain Type',
    'fbs':     'Fasting Blood Sugar > 120 mg/dl',
    'restecg': 'Resting ECG Results',
    'exang':   'Exercise Induced Angina',
    'slope':   'Slope of ST Segment',
    'ca':      'Major Vessels Colored',
    'thal':    'Thalassemia Type',
    'target':  'Heart Disease Diagnosis'
}

n_cols = 3
n_rows = -(-len(columns) // n_cols)  # ceiling division

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[cat_labels.get(c, c) for c in df_categorical.columns],
    vertical_spacing=0.10,
    horizontal_spacing=0.08
)

for i, col in enumerate(df_categorical.columns):
    row = (i // n_cols) + 1
    col_idx = (i % n_cols) + 1
    color = premium_colors[i % len(premium_colors)]

    val_counts = df_categorical[col].value_counts().sort_values()
    labels = val_counts.index.astype(str).tolist()
    counts = val_counts.values.tolist()
    percentages = [f'{v/sum(counts)*100:.1f}%' for v in counts]

    fig.add_trace(
        go.Bar(
            x=counts,
            y=labels,
            orientation='h',
            marker=dict(
                color=counts,
                colorscale=[[0, '#E0E7FF'], [1, color]],
                line=dict(color='white', width=0.8)
            ),
            text=percentages,
            textposition='outside',
            textfont=dict(size=11, color='#475569'),
            hovertemplate=(
                f'<b>{cat_labels.get(col, col)}</b><br>'
                'Category: %{y}<br>Count: %{x}<extra></extra>'
            )
        ),
        row=row, col=col_idx
    )

fig.update_layout(
    height=950,
    width=1150,
    title_text='<b>Categorical Feature Distributions</b><br><sup>Value Counts with Percentage Labels</sup>',
    showlegend=False,
    margin=dict(l=60, r=80, t=110, b=50)
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9', title_text='Count')
fig.update_yaxes(showgrid=False)

for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=13, color='#334155', family='Inter, Arial')

fig.show()


* **Key Observations:**
  * The target variable is relatively balanced, with **54.5%** of patients diagnosed with heart disease (Target = 1) and **45.5%** healthy (Target = 0).
  * Sex distribution is skewed, with significantly more male patients (~68%) than female patients (~32%).
  * Type 0 chest pain (Typical Angina) is the most common category (~47%).


### 3.3. Target vs. Continuous Features
We examine how continuous variables vary across healthy (Target=0) and sick (Target=1) patients using bar charts of average values and side-by-side KDE distribution comparison.


In [46]:
"""
(Main Code)
fig, ax = plt.subplots(len(continuous), 2, figsize=(15,15))
for i, col in enumerate(continuous):
    sns.barplot(data=df,x="target",y=col,ax=ax[i,0])
    sns.kdeplot(data=df[df["target"]==0],x=col,fill=True,ax=ax[i,1])
    sns.kdeplot(data=df[df["target"]==1],x=col,fill=True,ax=ax[i,1])
plt.show()
"""

# ── Target vs Continuous Features (PLOTLY PRO VERSION) ──
from scipy.stats import gaussian_kde
import numpy as np

fig = make_subplots(
    rows=len(continuous), cols=2,
    subplot_titles=[title for col in continuous for title in (f'Mean {col} by Target', f'{col} Distribution (KDE)')],
    vertical_spacing=0.08,
    horizontal_spacing=0.20  # Increased horizontal spacing between bar and KDE charts
)

for i, col in enumerate(continuous):
    row = i + 1
    
    # 1. Bar plot: Mean by target
    means = df.groupby('target')[col].mean()
    fig.add_trace(
        go.Bar(
            x=[str(idx) for idx in means.index], y=means.values,
            marker_color=[premium_colors[0], premium_colors[1]],
            name=f'Mean {col}',
            text=[f'{val:.1f}' for val in means.values],
            textposition='auto',
            textfont=dict(color='white', size=12, family='Inter, Arial'),
            showlegend=False,
            hovertemplate='Target %{x}<br>Mean: %{y:.2f}<extra></extra>'
        ),
        row=row, col=1
    )
    
    # Update axes titles for the bar plot
    fig.update_yaxes(title_text='Mean Value', row=row, col=1, title_font=dict(size=11, color='gray'))
    fig.update_xaxes(title_text='Target', row=row, col=1, title_font=dict(size=11, color='gray'))
    
    # 2. KDE plot: Density by target
    for target_val, color in zip([0, 1], [premium_colors[0], premium_colors[1]]):
        data = df[df['target'] == target_val][col].dropna()
        if len(data) > 1:
            kde = gaussian_kde(data)
            x_min, x_max = data.min(), data.max()
            padding = (x_max - x_min) * 0.1
            x_vals = np.linspace(x_min - padding, x_max + padding, 200)
            y_vals = kde(x_vals)
            
            fig.add_trace(
                go.Scatter(
                    x=x_vals, y=y_vals,
                    mode='lines',
                    line=dict(color=color, width=2.5),
                    fill='tozeroy',
                    opacity=0.6,
                    name=f'Target {target_val}',
                    showlegend=(i == 0), # Show legend only for the first row
                    hovertemplate=f'<b>Target {target_val}</b><br>Value: %{{x:.1f}}<br>Density: %{{y:.4f}}<extra></extra>'
                ),
                row=row, col=2
            )
            
            # Add a vertical dashed line for the mean value on the KDE plot
            mean_val = means[target_val]
            fig.add_vline(
                x=mean_val, line_width=1.5, line_dash="dash", line_color=color, 
                opacity=0.8, row=row, col=2,
                annotation_text=f"Mean: {mean_val:.1f}", 
                annotation_position="top left" if target_val == 0 else "top right",
                annotation_font=dict(size=10, color=color)
            )

    # Update axes titles for the KDE plot
    fig.update_xaxes(title_text=col, row=row, col=2, title_font=dict(size=11, color='gray'))
    fig.update_yaxes(title_text='Density', row=row, col=2, title_font=dict(size=11, color='gray'))

fig.update_layout(
    height=350 * len(continuous),
    width=1150,
    title_text='<b>Target vs Continuous Features</b>',
    margin=dict(l=50, r=30, t=100, b=70),
    plot_bgcolor='white',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Fix grid styling
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9', zeroline=False)

fig.show()

* **Key Observations:**
  * **Age & Trestbps:** Interestingly, the mean age and resting blood pressure are slightly lower in patients diagnosed with heart disease.
  * **Thalach:** Patients with heart disease have a significantly higher mean maximum heart rate (~158 bpm) compared to healthy patients (~139 bpm). This is a strong potential feature for classification.
  * **Oldpeak:** Healthy patients show much higher ST depression values compared to heart disease patients, suggesting lower ST depression correlates with heart disease presence in this sample.


### 3.4. Target vs. Categorical Features
We cross-tabulate target outcomes with categorical features using stacked bar charts to find key associations.


In [47]:
"""
(Main Code)
fig,ax = plt.subplots(3,3,figsize=(15, 15))
plot_columns = [col for col in columns if col != 'target']
for i, col in enumerate(plot_columns):
    x=i//3
    y=i%3
    pd.crosstab(df[col], df['target']).plot(kind='bar', stacked=True, ax=ax[x,y])
plt.show()
"""

# ── Target vs Categorical Features (PLOTLY PRO VERSION) ──
plot_columns = [col for col in columns if col != 'target']

n_cols = 3
n_rows = -(-len(plot_columns) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[cat_labels.get(c, c) for c in plot_columns],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, col in enumerate(plot_columns):
    row = (i // n_cols) + 1
    col_idx = (i % n_cols) + 1
    
    ct = pd.crosstab(df[col], df['target'])
    
    for j, target_val in enumerate(ct.columns):
        fig.add_trace(
            go.Bar(
                x=ct.index.astype(str),
                y=ct[target_val],
                name=f'Target {target_val}',
                marker_color=premium_colors[j],
                showlegend=(i == 0) # Only show legend once
            ),
            row=row, col=col_idx
        )

fig.update_layout(
    barmode='stack',
    height=1000,
    width=1150,
    title_text='<b>Target vs Categorical Features</b>',
    margin=dict(l=50, r=30, t=100, b=50),
    plot_bgcolor='white'
)
fig.show()

* **Key Observations:**
  * **Sex:** Female patients have a significantly higher proportion of heart disease diagnoses (~75%) compared to male patients (~45%) in this dataset.
  * **Chest Pain (cp):** Patients experiencing atypical angina (Type 1) or non-anginal pain (Type 2) show extremely high proportions of heart disease.
  * **Exercise Angina (exang):** Patients without exercise-induced angina (exang = 0) are much more likely to have heart disease.


## 4. Preprocessing & Outlier Treatment

### 4.1. Outlier Detection & Capping
We calculate the number of outliers in continuous columns using the Interquartile Range (IQR) method, and then treat them by capping (clipping) values to the 1.5 * IQR boundaries.


In [48]:
Q1 = df[continuous].quantile(0.25)
Q3 = df[continuous].quantile(0.75)
IQR = Q3 - Q1 
outliers = ((df[continuous] < (Q1-1.5*IQR)) | (df[continuous] > (Q3+1.5*IQR))).sum()
print("Outliers count before capping:")
print(outliers)

# Treat outliers by capping (clipping) them at the 1.5 * IQR boundaries
for col in continuous:
    lower_bound = Q1[col] - 1.5 * IQR[col]
    upper_bound = Q3[col] + 1.5 * IQR[col]
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

outliers_after = ((df[continuous] < (Q1-1.5*IQR)) | (df[continuous] > (Q3+1.5*IQR))).sum()
print("\nOutliers count after capping:")
print(outliers_after)


Outliers count before capping:
age         0
trestbps    9
chol        5
thalach     1
oldpeak     5
dtype: int64

Outliers count after capping:
age         0
trestbps    0
chol        0
thalach     0
oldpeak     0
dtype: int64


### 4.2. One-Hot Encoding
To prepare the dataset for machine learning models, we one-hot encode nominal multi-class columns (`cp`, `restecg`, `thal`) while dropping the first category to avoid multicollinearity. Binary variables are cast to integer format.


In [49]:
df_encoded = pd.get_dummies(df,columns=['cp','restecg','thal'], drop_first=True)

features_to_convert = ['sex', 'fbs', 'exang', 'slope', 'ca', 'target']
for feature in features_to_convert:
    df_encoded[feature] = df_encoded[feature].astype(int)

In [50]:
df_encoded.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,slope,ca,target,cp_1,cp_2,cp_3,restecg_1,restecg_2,thal_1,thal_2,thal_3
0,63,1,145,233.0,1,150.0,0,2.3,0,0,1,False,False,True,False,False,True,False,False
1,37,1,130,250.0,0,187.0,0,3.5,0,0,1,False,True,False,True,False,False,True,False
2,41,0,130,204.0,0,172.0,0,1.4,2,0,1,True,False,False,False,False,False,True,False
3,56,1,120,236.0,0,178.0,0,0.8,2,0,1,True,False,False,True,False,False,True,False
4,57,0,120,354.0,0,163.0,1,0.6,2,0,1,False,False,False,True,False,False,True,False


### 4.3. Train-Test Split
We split the data into training (80%) and testing (20%) sets. We use `stratify=y` to ensure the class distribution of heart disease is preserved across both splits.


In [51]:
x = df_encoded.drop('target',axis=1)
y = df_encoded['target']
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

## 5. Model Training & Hyperparameter Tuning
We implement a standardized, cross-validated hyperparameter tuning process. We use `StratifiedKFold` to maintain class distributions and optimize specifically for **Recall** (Class 1) to minimize dangerous false negatives.


In [52]:
def tune(clf, param_grid, x_train, y_train, scoring='recall', n_splits=3):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    grid = GridSearchCV(clf, param_grid, cv=cv, scoring=scoring, n_jobs=-1)
    grid.fit(x_train,y_train)
    best_hyperparamters = grid.best_params_
    return grid.best_estimator_ , best_hyperparamters

In [53]:
def evaluate(model,x_test,y_test,model_name):
    y_pred = model.predict(x_test)
    report = classification_report(y_test,y_pred, output_dict=True)
    metrics = {
        "precision_0": report["0"]["precision"],
        "precision_1": report["1"]["precision"],
        "recall_0": report["0"]["recall"],
        "recall_1": report["1"]["recall"],
        "f1_0": report["0"]["f1-score"],
        "f1_1": report["1"]["f1-score"],
        "macro_avg_precision": report["macro avg"]["precision"],
        "macro_avg_recall": report["macro avg"]["recall"],
        "macro_avg_f1": report["macro avg"]["f1-score"],
        "accuracy": accuracy_score(y_test, y_pred)
    }

    evaluation = pd.DataFrame(metrics, index=[model_name]).round(2)
    return evaluation

def evaluate_train(model,x_train,y_train,model_name):
    y_pred = model.predict(x_train)
    report = classification_report(y_train,y_pred, output_dict=True)
    metrics = {
        "precision_0": report["0"]["precision"],
        "precision_1": report["1"]["precision"],
        "recall_0": report["0"]["recall"],
        "recall_1": report["1"]["recall"],
        "f1_0": report["0"]["f1-score"],
        "f1_1": report["1"]["f1-score"],
        "macro_avg_precision": report["macro avg"]["precision"],
        "macro_avg_recall": report["macro avg"]["recall"],
        "macro_avg_f1": report["macro avg"]["f1-score"],
        "accuracy": accuracy_score(y_train, y_pred)
    }

    evaluation = pd.DataFrame(metrics, index=[model_name]).round(2)
    return evaluation

### 5.1. Decision Tree Classifier
We tune a Decision Tree Classifier, checking gini/entropy criteria and testing a range of depths to prevent under/overfitting.


In [54]:
dt = DecisionTreeClassifier(random_state=42)
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4, 5],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3]
}

best_model_dt, best_para_dt = tune(dt,param_grid_dt,x_train,y_train)
dt_evaluation = evaluate(best_model_dt,x_test,y_test,'DT_Test')
dt_evaluation_train = evaluate_train(best_model_dt,x_train,y_train,'DT_Train')
both = [dt_evaluation, dt_evaluation_train]
overfitting = pd.concat(both)
overfitting

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
DT_Test,0.67,0.68,0.57,0.76,0.62,0.71,0.67,0.66,0.66,0.67
DT_Train,0.85,0.89,0.87,0.87,0.86,0.88,0.87,0.87,0.87,0.87


### 5.2. Random Forest Classifier
We tune an ensemble Random Forest Classifier, searching over forest size, tree depth, splitting criteria, and bootstrap settings.


In [55]:
rf = RandomForestClassifier(random_state=42)
param_grid_rf = {
    'n_estimators': [10, 30, 50, 70, 100],
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3],
    'bootstrap': [True, False]
}

best_model_rf, best_para_rf = tune(rf,param_grid_rf,x_train,y_train)
rf_evaluation = evaluate(best_model_rf,x_test,y_test,'RF_Test')
rf_evaluation_train = evaluate_train(best_model_rf,x_train,y_train,'RF_Train')
both = [rf_evaluation, rf_evaluation_train]
overfitting = pd.concat(both)
overfitting

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
RF_Test,0.86,0.77,0.68,0.91,0.76,0.83,0.82,0.79,0.80,0.80
RF_Train,0.91,0.88,0.85,0.93,0.88,0.90,0.90,0.89,0.89,0.89


### 5.3. K-Nearest Neighbors (KNN)
We tune a K-Nearest Neighbors Classifier. Because KNN is highly sensitive to feature scales (distance-based), we wrap the preprocessing step (`StandardScaler`) and the model inside a Scikit-Learn `Pipeline` to prevent data leakage during cross-validation.


In [56]:
knn_pipeline = Pipeline([
    ('Scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid_knn = {
    'knn__n_neighbors': list(range(1, 12)),
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

best_model_knn, best_para_knn = tune(knn_pipeline, param_grid_knn, x_train, y_train)
knn_evaluation = evaluate(best_model_knn,x_test,y_test,'KNN_Test')
knn_evaluation_train = evaluate_train(best_model_knn,x_train,y_train,'KNN_Train')
both = [knn_evaluation, knn_evaluation_train]
overfitting = pd.concat(both)
overfitting

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
KNN_Test,0.77,0.77,0.71,0.82,0.74,0.79,0.77,0.77,0.77,0.77
KNN_Train,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


* **Overfitting Analysis — KNN:**
  * KNN achieves **100% training accuracy**, which is a classic sign of overfitting. This occurs because KNN with a small `k` value essentially memorizes the training data — each point is its own nearest neighbor.
  * The **18-point gap** between training recall (1.00) and test recall (0.82) is the largest across all models, confirming poor generalization.
  * With only **303 samples**, the training set (~242 samples) is small enough that KNN can perfectly partition the feature space, but this doesn't generalize to unseen data.
  * Despite GridSearchCV selecting the optimal `k`, the fundamental limitation is the **small dataset size** — there aren't enough neighbors to form robust decision boundaries. This is a known weakness of KNN on small, high-dimensional datasets.


### 5.4. Support Vector Machine (SVM)
We tune a Support Vector Classifier (SVC). Like KNN, we scale the inputs within a `Pipeline` to prevent leakage. We use standard kernels (linear, rbf, poly) and tune regularizations.


In [57]:
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True)) 
])

param_grid_svm = {
    'svm__C': [0.0011, 0.005, 0.01, 0.05, 0.1, 1, 10, 20],
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__gamma': ['scale', 'auto', 0.1, 0.5, 1, 5],  
    'svm__degree': [2, 3, 4]
}

best_model_svm, best_param_svm = tune(svm_pipeline, param_grid_svm, x_train, y_train)
svm_evaluation = evaluate(best_model_svm, x_test, y_test, 'SVM_Test')
svm_evaluation_train = evaluate_train(best_model_svm,x_train,y_train,'SVM_Train')
both = [svm_evaluation, svm_evaluation_train]
overfitting = pd.concat(both)
overfitting

,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
SVM_Test,0.86,0.66,0.43,0.94,0.57,0.78,0.76,0.68,0.67,0.70
SVM_Train,0.91,0.76,0.65,0.95,0.76,0.84,0.84,0.80,0.80,0.81


### 5.5. Gradient Boosting Classifier
We add a Gradient Boosting Classifier to test an iterative boosting ensemble method, tuning learning rate, estimators, and tree depth.


In [58]:
gb = GradientBoostingClassifier(random_state=42)
param_grid_gb = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 150],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0]
}

best_model_gb, best_param_gb = tune(gb, param_grid_gb, x_train, y_train)
gb_evaluation = evaluate(best_model_gb, x_test, y_test, 'GB_Test')
gb_evaluation_train = evaluate_train(best_model_gb,x_train,y_train,'GB_Train')
both = [gb_evaluation, gb_evaluation_train]
overfitting = pd.concat(both)
overfitting


,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
GB_Test,0.85,0.73,0.61,0.91,0.71,0.81,0.79,0.76,0.76,0.77
GB_Train,0.91,0.84,0.78,0.94,0.84,0.89,0.88,0.86,0.86,0.87


### 5.6. Stacking Classifier
To leverage the strengths of our diverse models (Decision Tree, Random Forest, KNN, SVM, and Gradient Boosting), we implement a **Stacking Classifier**. 
We use these previously tuned models as our base estimators and a `LogisticRegression` model as the final estimator. We tune the `C` hyperparameter of the final estimator using grid search to optimize for our primary metric: **Recall for Class 1**.

In [59]:
estimators = [
    ('dt', best_model_dt),
    ('rf', best_model_rf),
    ('knn', best_model_knn),
    ('svm', best_model_svm),
    ('gb', best_model_gb)
]

stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5)
param_grid_stack = {
    'final_estimator__C': [0.1, 1.0, 10.0]
}

best_model_stack, best_param_stack = tune(stack, param_grid_stack, x_train, y_train)
stack_evaluation = evaluate(best_model_stack, x_test, y_test, 'Stacking_Test')
stack_evaluation_train = evaluate_train(best_model_stack, x_train, y_train, 'Stacking_Train')
both = [stack_evaluation, stack_evaluation_train]
overfitting = pd.concat(both)
overfitting


,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
Stacking_Test,0.83,0.76,0.68,0.88,0.75,0.82,0.79,0.78,0.78,0.79
Stacking_Train,0.97,0.93,0.92,0.98,0.94,0.96,0.95,0.95,0.95,0.95


## 6. Model Evaluation & Comparison
We compare all models in a unified table, sorted by our primary metric, **Recall (Class 1)**.


In [60]:
all_evaluations = [dt_evaluation, rf_evaluation, knn_evaluation, svm_evaluation, gb_evaluation, stack_evaluation]
results = pd.concat(all_evaluations)

results = results.sort_values(by='recall_1', ascending=False).round(2)
results


,precision_0,precision_1,recall_0,recall_1,f1_0,f1_1,macro_avg_precision,macro_avg_recall,macro_avg_f1,accuracy
SVM_Test,0.86,0.66,0.43,0.94,0.57,0.78,0.76,0.68,0.67,0.70
RF_Test,0.86,0.77,0.68,0.91,0.76,0.83,0.82,0.79,0.80,0.80
GB_Test,0.85,0.73,0.61,0.91,0.71,0.81,0.79,0.76,0.76,0.77
Stacking_Test,0.83,0.76,0.68,0.88,0.75,0.82,0.79,0.78,0.78,0.79
KNN_Test,0.77,0.77,0.71,0.82,0.74,0.79,0.77,0.77,0.77,0.77
DT_Test,0.67,0.68,0.57,0.76,0.62,0.71,0.67,0.66,0.66,0.67


We visualize the Recall values for Class 1 (Heart Disease) across all models to select our champion model.


In [61]:
"""
(Main Code)
results = pd.concat([dt_evaluation, rf_evaluation, knn_evaluation, svm_evaluation, gb_evaluation, stack_evaluation])
results['recall_1'].sort_values().plot(kind='barh')
plt.show()
"""

# ── Model Comparison (PLOTLY PRO VERSION) ──
try:
    results = pd.concat([dt_evaluation, rf_evaluation, knn_evaluation, svm_evaluation, gb_evaluation, stack_evaluation])
    results_sorted = results.sort_values(by='recall_1')

    fig = go.Figure(go.Bar(
        x=results_sorted['recall_1'],
        y=results_sorted.index,
        orientation='h',
        marker=dict(
            color=results_sorted['recall_1'],
            colorscale='Viridis',
            line=dict(color='white', width=1)
        ),
        text=[f'{val:.3f}' for val in results_sorted['recall_1']],
        textposition='outside'
    ))

    fig.update_layout(
        title='<b>Model Comparison — Recall (Class 1)</b>',
        xaxis_title='Recall (Class 1)',
        yaxis_title='Model',
        height=450,
        width=800,
        margin=dict(l=100, r=50, t=80, b=50),
        plot_bgcolor='white'
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9')
    fig.show()
except NameError:
    print("Run the models above first to get evaluation results.")


## 7. Champion Model Diagnostics & Interpretability
We analyze our champion model (typically the model with the highest recall_1) in depth. We plot its **Confusion Matrix** to inspect classification trade-offs and evaluate its feature importances to verify clinical reasoning.


In [62]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

# 1. Confusion Matrix for Champion Model
best_model_name = results['recall_1'].idxmax()
model_map = {
    'DT_Test': best_model_dt,
    'RF_Test': best_model_rf,
    'KNN_Test': best_model_knn,
    'SVM_Test': best_model_svm,
    'GB_Test': best_model_gb,
    'Stacking_Test': best_model_stack
}
champion_model = model_map[best_model_name]

print(f"Champion Model Selected: {best_model_name}")

y_pred = champion_model.predict(x_test)
cm = confusion_matrix(y_test, y_pred)

fig_cm = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x="Predicted Label", y="True Label", color="Count"),
    x=['Healthy (0)', 'Heart Disease (1)'],
    y=['Healthy (0)', 'Heart Disease (1)'],
    color_continuous_scale='Blues',
    title=f'<b>Confusion Matrix — Champion Model ({best_model_name})</b>'
)
fig_cm.update_layout(
    width=500,
    height=450,
    margin=dict(l=50, r=50, t=80, b=50),
    coloraxis_showscale=False
)
fig_cm.show()

# 2. Feature Importance from Random Forest/Gradient Boosting reference
# Since SVM is RBF kernel, we will use the Random Forest or Gradient Boosting model to show feature importances
if hasattr(best_model_rf, 'feature_importances_'):
    importances = best_model_rf.feature_importances_
    features = x.columns
    df_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=True)

    fig_imp = go.Figure(go.Bar(
        x=df_imp['Importance'],
        y=df_imp['Feature'],
        orientation='h',
        marker=dict(color=df_imp['Importance'], colorscale='Plasma')
    ))
    fig_imp.update_layout(
        title='<b>Feature Importances (Random Forest Reference)</b><br><sup>Relative contribution of features to prediction</sup>',
        xaxis_title='Importance Score',
        yaxis_title='Feature',
        width=800,
        height=500,
        margin=dict(l=150, r=50, t=80, b=50),
        plot_bgcolor='white'
    )
    fig_imp.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9')
    fig_imp.show()


Champion Model Selected: SVM_Test


### 7.2. ROC-AUC Curves
We plot the **Receiver Operating Characteristic (ROC)** curve for all models. The ROC curve illustrates the trade-off between True Positive Rate (Sensitivity) and False Positive Rate (1 - Specificity) across all classification thresholds. A higher AUC (Area Under the Curve) indicates better overall discriminatory power.


In [63]:
from sklearn.metrics import roc_curve, auc

fig = go.Figure()
models = {'DT': best_model_dt, 'RF': best_model_rf, 'KNN': best_model_knn, 'SVM': best_model_svm, 'GB': best_model_gb, 'Stacking': best_model_stack}

for i, (name, model) in enumerate(models.items()):
    y_prob = model.predict_proba(x_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'{name} (AUC={roc_auc:.3f})', line=dict(color=premium_colors[i], width=2.5)))

fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash', color='gray', width=1), name='Random Baseline', showlegend=True))
fig.update_layout(
    title='<b>ROC Curves — All Models</b><br><sup>Higher AUC = Better Discriminatory Power</sup>',
    xaxis_title='False Positive Rate (1 - Specificity)',
    yaxis_title='True Positive Rate (Sensitivity)',
    width=750, height=550,
    plot_bgcolor='white',
    legend=dict(x=0.55, y=0.05)
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F1F5F9')
fig.show()


* **ROC-AUC Observations:**
  * All models perform significantly better than the random baseline (diagonal), confirming learned predictive patterns.
  * Models with higher AUC values demonstrate stronger ability to distinguish between healthy and heart disease patients across all possible thresholds.
  * The ROC-AUC complements our recall-focused evaluation — a model with high recall but low AUC may only perform well at one specific threshold, while high AUC indicates robust performance across thresholds.


## 8. Summary & Conclusion
* **Key Achievements:**
  * Implemented an end-to-end Machine Learning pipeline to identify patient heart disease.
  * Successfully identified and treated outliers using IQR clipping to stabilize model parameters.
  * Optimized hyperparameter tuning using cross-validation targeted at **Recall (Class 1)** to minimize dangerous false negatives in diagnostic recommendations.
  * Built and compared five machine learning algorithms: Decision Trees, Random Forests, K-Nearest Neighbors, Support Vector Machines, and Gradient Boosting.
  * Analyzed champion model classification errors and validated relative clinical feature weights (e.g. chest pain type, max heart rate, thalassemia type).


## 9. Limitations & Future Work
While the models performed well on this specific dataset, several limitations must be acknowledged:
* **Dataset Size:** The dataset contains only 303 instances, which is relatively small for deep generalization. More data is required to make the model clinically robust.
* **Feature Scope:** Crucial features such as patient family history, BMI, smoking habits, and lifestyle metrics are not included, which could significantly improve predictive power.
* **Demographic Representation:** The data may not perfectly represent the wider population (e.g., potential gender or age biases in the sample).
* **Clinical Validation Needed:** A machine learning model in healthcare is meant to augment, not replace, medical professionals. This model requires rigorous external validation on independent, larger cohorts before considering any real-world clinical deployment.

### Exporting the Champion Model
To deploy our trained model in a real-world application (such as a Streamlit web app), we need to serialize it. Here, we export the dynamically chosen `champion_model` using the `pickle` library. This creates a `.sav` file that can be instantly loaded later for inference without needing to retrain the pipeline.

In [64]:
import pickle

# Save the champion model to a file
filename = 'heart_disease_model.sav'
pickle.dump(champion_model, open(filename, 'wb'))
print(f"Model saved successfully as {filename}")


Model saved successfully as heart_disease_model.sav
